# Importação das Bibliotecas necessárias

In [1]:
from ultralytics import YOLO
import os
from IPython.display import display, Image
from IPython import display
import requests
import zipfile
import shutil
import time
display.clear_output()


# Donwload do DataSet

In [2]:
# URL do arquivo zip no Google Drive
drive_url = "https://drive.usercontent.google.com/download?id=1VGnPFfXVZcLY2-fT3M7zXKYTPXMSAaja&export=download&authuser=0&confirm=t&uuid=56adfbe7-4b31-43b4-a026-f07601384fcb&at=AN8xHooAXPh68VgPFMRtWTtDg3Wo%3A1758289282777"

# Nome do arquivo zip local
zip_filename = "dataset.zip"

# Pasta de destino
dest_dir = "Data//fruit.v11i.yolov11"

print("VERIFICANDO DATASET EXISTENTE...")
print("=" * 50)

# Verificar se o dataset já existe
dataset_exists = False
if os.path.exists(dest_dir):
    # Verificar se contém dados válidos
    subdirs = [d for d in os.listdir(dest_dir) if os.path.isdir(os.path.join(dest_dir, d))]
    
    if subdirs:
        print(f"Pasta 'Data' encontrada com {len(subdirs)} subpasta(s):")
        for subdir in subdirs:
            subdir_path = os.path.join(dest_dir, subdir)
            
            # Contar arquivos na subpasta
            total_files = 0
            for root, dirs, files in os.walk(subdir_path):
                total_files += len(files)
            
            print(f"    {subdir} - {total_files} arquivos")
        
        dataset_exists = True
        
        # Verificar especificamente o dataset de frutas

        if os.path.exists(dest_dir):
            data_yaml = os.path.join(dest_dir, "data.yaml")
            if os.path.exists(data_yaml):
                print(f"Dataset de frutas completo encontrado!")
            else:
                print(f"Dataset de frutas incompleto (falta data.yaml)")
        else:
            print(f"Dataset de frutas não encontrado")
    else:
        print(f"Pasta 'Data' existe mas está vazia")
else:
    print(f"Pasta 'Data' não encontrada")

# Decidir se deve fazer download
should_download = False

if dataset_exists:
    print(f"\nCONFIRMAÇÃO NECESSÁRIA:")
    print(f"Um dataset já foi encontrado na pasta 'Data'.")
    print(f"")
    print(f"Opções:")
    print(f" Pular download (usar dataset existente)")
    print(f" Baixar novamente (sobrescrever)")
    print(f" Baixar em nova pasta (manter ambos)")
    
    choice = input(f"\nDigite sua escolha (1/2/3): ").strip()
    
    if choice == "1":
        print(f"Usando dataset existente!")
        should_download = False
    elif choice == "2":
        print(f"Sobrescrevendo dataset existente...")
        should_download = True
        # Limpar pasta existente
        if os.path.exists(dest_dir):
            shutil.rmtree(dest_dir)
    elif choice == "3":
        print(f"Criando nova pasta para o dataset...")
        should_download = True
        dest_dir = f"Data_new_{int(time.time())}"  # Pasta única
        print(f"   Nova pasta: {dest_dir}")
    else:
        print(f"Opção inválida. Cancelando download.")
        should_download = False
else:
    print(f"\nDataset não encontrado. Iniciando download...")
    should_download = True

# Executar download se necessário
if should_download:
    try:
        print(f"\nBaixando dataset...")
        print(f"URL: {drive_url[:50]}...")
        print(f"Destino: {dest_dir}")
        
        # Download do arquivo zip com barra de progresso
        response = requests.get(drive_url, stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        downloaded_size = 0
        
        with open(zip_filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded_size += len(chunk)
                    
                    # Mostrar progresso simples
                    if total_size > 0:
                        progress = (downloaded_size / total_size) * 100
                        print(f"\rProgresso: {progress:.1f}% ({downloaded_size/(1024*1024):.1f}MB)", end="")
        
        print(f"\nDownload concluído!")
        
        # Extrair o zip
        print(f"Extraindo arquivos...")
        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            # Extrai para uma pasta temporária
            temp_extract_dir = "temp_dataset"
            zip_ref.extractall(temp_extract_dir)
        
        # Criar pasta de destino se não existir
        os.makedirs(dest_dir, exist_ok=True)
        
        # Mover conteúdo extraído para a pasta de destino
        for item in os.listdir(temp_extract_dir):
            s = os.path.join(temp_extract_dir, item)
            d = os.path.join(dest_dir, item)
            if os.path.isdir(s):
                if os.path.exists(d):
                    shutil.rmtree(d)
                shutil.move(s, d)
            else:
                shutil.move(s, d)
        
        # Limpar arquivos temporários
        shutil.rmtree(temp_extract_dir)
        os.remove(zip_filename)
        
        print(f"Dataset extraído com sucesso!")
        print(f"Localização: {os.path.abspath(dest_dir)}")
        
    except Exception as e:
        print(f"Erro durante o download: {e}")
        # Limpar arquivos em caso de erro
        if os.path.exists(zip_filename):
            os.remove(zip_filename)
        if os.path.exists("temp_dataset"):
            shutil.rmtree("temp_dataset")
else:
    print(f"\nProsseguindo com dataset existente...")
    print(f"Localização: {os.path.abspath(dest_dir)}")

VERIFICANDO DATASET EXISTENTE...
Pasta 'Data' encontrada com 3 subpasta(s):
    test - 1392 arquivos
    train - 30037 arquivos
    valid - 2819 arquivos
Dataset de frutas completo encontrado!

CONFIRMAÇÃO NECESSÁRIA:
Um dataset já foi encontrado na pasta 'Data'.

Opções:
 Pular download (usar dataset existente)
 Baixar novamente (sobrescrever)
 Baixar em nova pasta (manter ambos)
Usando dataset existente!

Prosseguindo com dataset existente...
Localização: c:\Users\gcald\Área de Trabalho\dev\Classificao-de-Alimentos-YOLO\Data\fruit.v11i.yolov11


# Donwload de Modelos para Treinamento

In [3]:
# Lista de todos os modelos YOLOv11 disponíveis
yolo_models = {
    'yolo11n.pt': 'Nano - Mais rápido, menor precisão (~6MB)',
    'yolo11s.pt': 'Small - Balanceado velocidade/precisão (~22MB)', 
    'yolo11m.pt': 'Medium - Boa precisão, velocidade moderada (~50MB)',
    'yolo11l.pt': 'Large - Alta precisão, mais lento (~52MB)',
    'yolo11x.pt': 'Extra Large - Máxima precisão, mais lento (~138MB)'
}

models_downloaded = []
models_dir = "Models"

# Criar diretório se não existir
os.makedirs(models_dir, exist_ok=True)

for model_name, description in yolo_models.items():
    print(f"\nBaixando {model_name}...")
    print(f"   {description}")
    
    try:
        start_time = time.time()
        
        # Caminho completo do modelo
        model_path = os.path.join(models_dir, model_name)
        
        # Carregar modelo (isso fará o download se necessário)
        model = YOLO(model_path)
        
        download_time = time.time() - start_time
        
        # Verificar tamanho do arquivo
        if os.path.exists(model_path):
            file_size = os.path.getsize(model_path) / (1024*1024)  # MB
            print(f"Sucesso! Tamanho: {file_size:.1f} MB")
            print(f"Tempo: {download_time:.1f}s")
            models_downloaded.append(model_name)
        else:
            print(f"Modelo pode estar no cache do ultralytics")
            models_downloaded.append(model_name)
            
    except Exception as e:
        print(f"Erro ao baixar {model_name}: {e}")


print(f"Modelos baixados com sucesso: {len(models_downloaded)}")
for model in models_downloaded:
    print(f"  {model}")

print(f"\nLocalização: {os.path.abspath(models_dir)}")
print("Todos os modelos estão prontos para treinamento!")


Baixando yolo11n.pt...
   Nano - Mais rápido, menor precisão (~6MB)
Sucesso! Tamanho: 5.4 MB
Tempo: 0.0s

Baixando yolo11s.pt...
   Small - Balanceado velocidade/precisão (~22MB)
Sucesso! Tamanho: 18.4 MB
Tempo: 0.0s

Baixando yolo11m.pt...
   Medium - Boa precisão, velocidade moderada (~50MB)
Sucesso! Tamanho: 38.8 MB
Tempo: 0.0s

Baixando yolo11l.pt...
   Large - Alta precisão, mais lento (~52MB)
Sucesso! Tamanho: 49.0 MB
Tempo: 0.1s

Baixando yolo11x.pt...
   Extra Large - Máxima precisão, mais lento (~138MB)
Sucesso! Tamanho: 109.3 MB
Tempo: 0.1s
Modelos baixados com sucesso: 5
  yolo11n.pt
  yolo11s.pt
  yolo11m.pt
  yolo11l.pt
  yolo11x.pt

Localização: c:\Users\gcald\Área de Trabalho\dev\Classificao-de-Alimentos-YOLO\Models
Todos os modelos estão prontos para treinamento!


# Verificação do Ambiente e Dependências

In [4]:
# Verificar versões das bibliotecas importantes
import torch
import cv2
import numpy as np
import sys
from ultralytics import __version__ as ultralytics_version

print("=== INFORMAÇÕES DO AMBIENTE ===")
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Ultralytics: {ultralytics_version}")

print("\n=== RECURSOS DE HARDWARE ===")
print(f"CPU Cores: {os.cpu_count()}")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Memória GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA não disponível - treinamento será feito na CPU")

=== INFORMAÇÕES DO AMBIENTE ===
Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)]
PyTorch: 2.6.0+cu124
OpenCV: 4.12.0
NumPy: 2.2.6
Ultralytics: 8.3.193

=== RECURSOS DE HARDWARE ===
CPU Cores: 12
CUDA Disponível: True
GPU: NVIDIA GeForce RTX 4060
CUDA Version: 12.4
Memória GPU: 8.0 GB


## ANÁLISE DETALHADA DO DATASET

In [5]:
# Análise detalhada do dataset de frutas
import yaml

print("=== ANÁLISE DETALHADA DO DATASET ===")

# Ler o arquivo data.yaml
data_yaml_path = r"Data/fruit.v11i.yolov11/data.yaml"
try:
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    print(f"Número de classes: {data_config.get('nc', 'Não especificado')}")
    print(f"Classes: {data_config.get('names', 'Não especificado')}")
    print(f"Caminho de treino: {data_config.get('train', 'Não especificado')}")
    print(f"Caminho de validação: {data_config.get('val', 'Não especificado')}")
    print(f"Caminho de teste: {data_config.get('test', 'Não especificado')}")
    
    # Calcular estatísticas
    total_images = 10714 + 707 + 655  # train + valid + test
    print(f"\nESTATÍSTICAS:")
    print(f"Total de imagens: {total_images}")
    print(f"Treino: {10714} ({10714/total_images*100:.1f}%)")
    print(f"Validação: {707} ({707/total_images*100:.1f}%)")
    print(f"Teste: {655} ({655/total_images*100:.1f}%)")
    
    print(f"\nDISTRIBUIÇÃO IDEAL:")
    print(f"Proporção treino/validação: {10714/707:.1f}:1 (boa)")
    print(f"Dataset bem balanceado para treinamento")
    
except Exception as e:
    print(f"Erro ao ler data.yaml: {e}")

print(f"\nESPAÇO EM DISCO:")
# Estimar tamanho do dataset (aproximado)
estimated_size = total_images * 0.2  # Assumindo ~200KB por imagem
print(f"Tamanho estimado das imagens: ~{estimated_size/1024:.1f} GB")

=== ANÁLISE DETALHADA DO DATASET ===
Número de classes: 18
Classes: ['Apple', 'Avocado', 'Banana', 'Coconut', 'Dragon-fruit', 'Durian', 'Guava', 'Jackfruit', 'Lychee', 'Mango', 'Mangosteen', 'Orange', 'Papaya', 'Pear', 'Pineapple', 'Pomegranate', 'Strawberry', 'Watermelon']
Caminho de treino: ../train/images
Caminho de validação: ../valid/images
Caminho de teste: ../test/images

ESTATÍSTICAS:
Total de imagens: 12076
Treino: 10714 (88.7%)
Validação: 707 (5.9%)
Teste: 655 (5.4%)

DISTRIBUIÇÃO IDEAL:
Proporção treino/validação: 15.2:1 (boa)
Dataset bem balanceado para treinamento

ESPAÇO EM DISCO:
Tamanho estimado das imagens: ~2.4 GB


# Treinamento do Modelo

In [ ]:
# Verificação automática de dispositivo disponível
import torch

print("VERIFICANDO DISPOSITIVOS DISPONÍVEIS...")
if torch.cuda.is_available():
    device = '0'  # GPU
    device_name = torch.cuda.get_device_name(0)
    print(f"Usando GPU: {device_name}")
else:
    device = 'cpu'  # CPU
    print(f"Usando CPU: {os.cpu_count()} cores")

print(f"Dispositivo selecionado: {device}")

# Carregando um modelo pré-treinado para treinamento
path_model = r"Models/yolo11n.pt"
dataset_path = r"Data/fruit.v11i.yolov11/data.yaml"

print(f"\nCarregando modelo: {path_model}")
print(f"Dataset: {dataset_path}")

model = YOLO(path_model)

# Configurações de treinamento ajustadas para CPU
print(f"\nINICIANDO TREINAMENTO...")
print(f"Tamanho da imagem: 640px")
print(f"Dispositivo: {device}")

# Treinamento do Modelo com configurações otimizadas para CPU
if not torch.cuda.is_available():
    model_train = model.train(
        data=dataset_path, 
        epochs=5, 
        batch=0.80,  # Reduzido para CPU
        imgsz=640, 
        name="yolo11n-fruit", 
        exist_ok=True, 
        device=device,  # Usa dispositivo detectado automaticamente
        workers=os.cpu_count() - 2,  # Número de workers para CPU
        patience=50,  # Paciência para early stopping
        save=True,  # Salvar checkpoints
        plots=True  # Gerar gráficos de treinamento
        
    )
#GPU
else: model_train = model.train(
    data=dataset_path,
    epochs=30,
    batch=0.90,       # <--- Batch maior
    imgsz=640,
    name="yolo11n-fruit",
    exist_ok=True,
    device=0,       
    workers=os.cpu_count() - 2,
    patience=50,
    save=True,
    plots=True
)
    

print("\nTREINAMENTO Concluído!")

VERIFICANDO DISPOSITIVOS DISPONÍVEIS...
Usando GPU: NVIDIA GeForce RTX 4060
Dispositivo selecionado: 0

Carregando modelo: Models/yolo11n.pt
Dataset: Data/fruit.v11i.yolov11/data.yaml

INICIANDO TREINAMENTO...
Tamanho da imagem: 640px
Dispositivo: 0
New https://pypi.org/project/ultralytics/8.3.233 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.193  Python-3.12.2 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Data/fruit.v11i.yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, i

# Validação do Treinamento do Modelo

In [ ]:
# Carregando um modelo
model = YOLO(r"runs\weights\best.pt")  

# Validando o modelo
metrics = model.val()
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # uma lista contem map50-95 de cada categoria

Ultralytics 8.3.193  Python-3.12.2 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
YOLO11n summary (fused): 100 layers, 2,585,662 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 489.3201.2 MB/s, size: 51.1 KB)
val: Scanning C:\Users\gcald\Área de Trabalho\dev\Classificao-de-Alimentos-YOLO\Data\fruit.v11i.yolov11\valid\labels.cache... 1409 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1409/1409 1.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 89/89 11.0it/s 8.1s0.2s
                   all       1409       2902      0.924      0.912      0.956      0.785
                 Apple        211        405      0.958      0.914      0.979      0.938
               Avocado         61        148      0.939      0.828      0.951      0.862
                Banana        168        199      0.947      0.915      0.952      0.817
               Coconut         67        155   

array([    0.93836,     0.86156,     0.81711,     0.65007,     0.72498,     0.73531,     0.75708,     0.85968,     0.82856,     0.73075,     0.73257,     0.79962,     0.86137,     0.89844,     0.61594,     0.70897,      0.9161,     0.69095])

# Testando o Modelo Final

In [ ]:
INPUT_VAL_FOLDER = r".\DataProd\Input"
OUTPUT_VAL_FOLDER = r".\DataProd\Output"

v_it = int(len(os.listdir(OUTPUT_VAL_FOLDER))) + 1
OUTPUT_VAL_FOLDER = os.path.join(OUTPUT_VAL_FOLDER, f"{v_it}")
os.makedirs(OUTPUT_VAL_FOLDER, exist_ok=True)

In [ ]:
from ultralytics import YOLO

model = YOLO(r"runs/detect/yolo11n-fruit/weights/best.pt")
print("Modelo carregado com sucesso!")
print(model.names)


Modelo carregado com sucesso!
{0: 'Apple', 1: 'Avocado', 2: 'Banana', 3: 'Coconut', 4: 'Dragon-fruit', 5: 'Durian', 6: 'Guava', 7: 'Jackfruit', 8: 'Lychee', 9: 'Mango', 10: 'Mangosteen', 11: 'Orange', 12: 'Papaya', 13: 'Pear', 14: 'Pineapple', 15: 'Pomegranate', 16: 'Strawberry', 17: 'Watermelon'}


In [ ]:
import os
import cv2
from PIL import Image

for file in os.listdir(INPUT_VAL_FOLDER):
    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):

        input_image_path = os.path.join(INPUT_VAL_FOLDER, file)
        output_image_path = os.path.join(OUTPUT_VAL_FOLDER, file)

        print(f"\nProcessando imagem: {input_image_path}")

        # Testa se a imagem pode ser lida
        img_test = cv2.imread(input_image_path)
        if img_test is None:
            print(f"ERRO: Imagem corrompida ou ilegível. Pulando → {file}")
            continue  # pula para a próxima imagem

        try:
            results = model.predict(
                source=input_image_path,
                save=False
            )
        except Exception as e:
            print(f"ERRO no YOLO ao processar {file}: {e}")
            continue
        
        # Converte de BGR para RGB
        img_bgr = results[0].plot()
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        Image.fromarray(img_rgb).save(output_image_path)
        print(f"✔ Resultado salvo em: {output_image_path}")



Processando imagem: .\DataProd\Input\0000000003421.jpg

image 1/1 c:\Users\gcald\rea de Trabalho\dev\Classificao-de-Alimentos-YOLO\DataProd\Input\0000000003421.jpg: 576x640 2 Watermelons, 79.7ms
Speed: 3.7ms preprocess, 79.7ms inference, 2.8ms postprocess per image at shape (1, 3, 576, 640)
✔ Resultado salvo em: .\DataProd\Output\2\0000000003421.jpg

Processando imagem: .\DataProd\Input\1.jpg

image 1/1 c:\Users\gcald\rea de Trabalho\dev\Classificao-de-Alimentos-YOLO\DataProd\Input\1.jpg: 480x640 2 Watermelons, 37.4ms
Speed: 5.5ms preprocess, 37.4ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)
✔ Resultado salvo em: .\DataProd\Output\2\1.jpg

Processando imagem: .\DataProd\Input\210312.jpg

image 1/1 c:\Users\gcald\rea de Trabalho\dev\Classificao-de-Alimentos-YOLO\DataProd\Input\210312.jpg: 384x640 1 Pineapple, 7.9ms
Speed: 1.8ms preprocess, 7.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
✔ Resultado salvo em: .\DataProd\Output\2\210312.jpg

P